# Notebook 02: Models — Debutanizer Soft Sensor
Stages 3, 4, 5, 6, and 7: evaluation setup, baseline, features, model training, and comparison.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_io import load_training_data
from src.features import build_causal_features, build_autoregressive_features
from src.evaluate import chronological_split, scale_features, calculate_metrics
from src.models import get_model_grids, run_experiment, generate_comparison_plots_and_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## Load Data

In [ ]:
df = load_training_data()
print("Dataset shape:", df.shape)


## Stage 3: Evaluation Setup (Chronological Split)
Why not random split? Random splitting leaks information between time steps.
A sample at time t+1 lands in training while time t is in test, giving the model
implicit access to the future, which unrealistically inflates performance scores.


In [ ]:
# Demonstrate the chronological split
X, y = build_causal_features(df, max_lag=4)
X_train, X_val, X_test, y_train, y_val, y_test = chronological_split(X, y)
print(f"Train: rows {X_train.index[0]}–{X_train.index[-1]}  ({len(X_train)} samples)")
print(f"Val:   rows {X_val.index[0]}–{X_val.index[-1]}  ({len(X_val)} samples)")
print(f"Test:  rows {X_test.index[0]}–{X_test.index[-1]}  ({len(X_test)} samples)")


## Stage 5: Feature Variants

In [ ]:
Xc4, yc4     = build_causal_features(df, max_lag=4)
Xc16, yc16   = build_causal_features(df, max_lag=16)
Xa4, ya4     = build_autoregressive_features(df, max_lag=4)
Xa16, ya16   = build_autoregressive_features(df, max_lag=16)

for name, X in [("Causal lag-4", Xc4), ("Causal lag-16", Xc16),
                ("Auto lag-4",  Xa4),  ("Auto lag-16",   Xa16)]:
    print(f"{name}: {X.shape[0]} rows × {X.shape[1]} features")


## Stage 6 & 7: Full Benchmark Run

In [ ]:
# This runs the full grid search across all 4 models and 4 variants (~5-10 min)
experiments = run_experiment(df)


In [ ]:
generate_comparison_plots_and_report(all_experiments=experiments)
print("Comparison report saved to results/model_comparison.md")


## Results Tables

In [ ]:
df_res = pd.read_csv("../results/model_results.csv")

for variant in df_res["feature_variant"].unique():
    print(f"\n{'='*60}")
    print(f"Variant: {variant}")
    sub = df_res[df_res["feature_variant"] == variant].sort_values("val_rmse")
    print(sub[["model_name","val_rmse","test_rmse","test_mae","test_r2","test_bias"]].to_string(index=False))


In [ ]:
# Show figures
from IPython.display import Image, display
for fig_name in ["test_rmse_comparison", "scatter_predicted_vs_actual",
                 "best_model_time_series", "best_model_residuals"]:
    path = f"../figures/{fig_name}.png"
    if os.path.exists(path):
        print(f"\n{fig_name}")
        display(Image(path, width=900))
